# Domain 3 — Claude Code (3.1%)

One skill: Claude Code Operation. This domain is configuration and
operational knowledge rather than API code — but it is still worth writing
the files out and reading them, because the exam asks about precedence and
about which component does what.

## 3.1 The CLAUDE.md hierarchy

CLAUDE.md files layer from broad to specific, with more specific scopes
taking precedence:

1. **Enterprise** — org-wide policy, managed centrally.
2. **Project** — committed to the repo, shared with the team. This is
   where the conventions a codebase depends on belong.
3. **Project-local** — your personal overrides for one repo, gitignored.
4. **User** — `~/.claude/CLAUDE.md`, your preferences everywhere.

Rule of thumb: if a teammate checking out the repo would need it, it goes
in the project file and gets committed.


In [ ]:
import json
import os

PROJECT_CLAUDE_MD = """\
# Claims Platform

## Stack
Python 3.11, FastAPI, Postgres. Anthropic SDK for all model calls.

## Model policy
- Pinned: `claude-haiku-4-5-20251001` for triage, `claude-sonnet-5` for
  letter drafting.
- Never use a floating alias in `src/production/`. Pin dated IDs so a
  model release cannot silently change behaviour.
- Upgrading a pin requires the eval suite in `evals/` to pass first.

## Conventions
- Type hints on every public function; `ruff` and `mypy` must pass.
- All I/O is async. Use `asyncio.gather` for independent calls.
- Model output is always validated against a pydantic model before use.
  Never trust a raw response body.

## Security
- Never put credentials or PII in prompt content. Redact before the call.
- Tool calls that write or pay go through `guardrails/policy.py`.
- No secrets in this file, in tests, or in fixtures.

## Where things live
- Skills:   `.claude/skills/`
- Commands: `.claude/commands/`
- Subagents:`.claude/agents/`
- Settings: `.claude/settings.json` (shared), `settings.local.json` (yours)
"""

os.makedirs(".claude", exist_ok=True)
with open("CLAUDE.md", "w") as handle:
    handle.write(PROJECT_CLAUDE_MD)

print(PROJECT_CLAUDE_MD)


## 3.2 settings.json

`settings.json` is committed and shared; `settings.local.json` is personal
and gitignored. The categories it governs are what the exam asks about:
permissions, hooks, environment, and model selection.

Permissions are the important one. `allow` auto-runs, `ask` prompts, and
`deny` blocks outright — and a deny rule is what keeps a destructive
command from ever being one accidental keystroke away.


In [ ]:
SETTINGS = {
    "model": "claude-sonnet-5",
    "permissions": {
        "allow": [
            "Read(src/**)",
            "Bash(pytest:*)",
            "Bash(ruff:*)",
        ],
        "ask": [
            "Write(src/**)",
            "Bash(git push:*)",
        ],
        "deny": [
            "Read(.env)",
            "Read(**/secrets/**)",
            "Bash(rm -rf:*)",
            "Bash(curl:*)",
        ],
    },
    "hooks": {
        "PreToolUse": [
            {
                "matcher": "Bash",
                "hooks": [
                    {
                        "type": "command",
                        "command": ".claude/hooks/check_command.sh",
                    }
                ],
            }
        ],
        "PostToolUse": [
            {
                "matcher": "Write|Edit",
                "hooks": [
                    {"type": "command", "command": "ruff check --fix"}
                ],
            }
        ],
    },
    "env": {"CLAIMS_API_BASE": "https://internal.example.com"},
}

with open(".claude/settings.json", "w") as handle:
    json.dump(SETTINGS, handle, indent=2)

print(json.dumps(SETTINGS, indent=2))


In [ ]:
HOOK_SCRIPT = """\
#!/usr/bin/env bash
# PreToolUse hook: receives the tool call as JSON on stdin.
# Exit 0 to allow, exit 2 to BLOCK and return stderr to the model.
set -euo pipefail

payload=$(cat)
command=$(echo "$payload" | jq -r '.tool_input.command // ""')

for forbidden in "rm -rf /" "DROP TABLE" "git push --force"; do
  if [[ "$command" == *"$forbidden"* ]]; then
    echo "BLOCKED: '$forbidden' is not permitted in this repo" >&2
    exit 2
  fi
done

exit 0
"""

os.makedirs(".claude/hooks", exist_ok=True)
with open(".claude/hooks/check_command.sh", "w") as handle:
    handle.write(HOOK_SCRIPT)
os.chmod(".claude/hooks/check_command.sh", 0o755)

print(HOOK_SCRIPT)
print("Exit code 2 is the one that matters: it blocks and tells the model "
      "why, so it can adapt instead of retrying blindly.")


## 3.3 The components, distinguished

These five get confused with each other, which is exactly why they are
tested:

| Component | What it is | Invoked by | Lives in |
|---|---|---|---|
| **Rules** (CLAUDE.md) | Standing context and conventions, always loaded | Nothing — always on | `CLAUDE.md` |
| **Skill** | Procedural knowledge, loaded on relevance | Claude, when relevant | `.claude/skills/` |
| **Command** | A prompt template you trigger explicitly | You, via `/name` | `.claude/commands/` |
| **Subagent** | A separate agent with its own context window | Claude, by delegation | `.claude/agents/` |
| **Agent Memory** | State persisted across sessions | Read and written by the agent | Memory store |

The distinction the exam presses: a **Command** is user-invoked, a **Skill**
is model-invoked. And a **Subagent** exists for context isolation — a fresh
window, not just a different prompt.


In [ ]:
COMMAND = """\
---
description: Review a claims module for schema-validation gaps
---

Review $ARGUMENTS for these specific issues:

1. Any `client.messages.create` result used without schema validation.
2. Any floating model alias in production code (must be a dated ID).
3. Any tool call that writes or pays without passing through
   `guardrails/policy.py`.
4. Any PII reaching a prompt without redaction.

Report findings as a numbered list with file:line references. Do not fix
anything yet.
"""

os.makedirs(".claude/commands", exist_ok=True)
with open(".claude/commands/review-claims.md", "w") as handle:
    handle.write(COMMAND)

print("Invoke with:  /review-claims src/triage.py")
print()
print(COMMAND)


## 3.4 Modes

- **Interactive** — the normal REPL session.
- **Headless** (`claude -p "prompt"`) — non-interactive, one shot, exits
  with a status code. This is the CI/CD answer: a pipeline has no terminal
  to type into, so a code-review or migration step runs headless and gates
  the build on its exit code.
- **Streaming** — emits output incrementally, for piping into another
  process or rendering live.
- **Auto-mode** — reduced approval prompting. Powerful and risky: pair it
  with tight `deny` permissions and hooks, because you have removed the
  human who was previously catching mistakes.

Session management matters for cost as much as correctness: a long session
carries its whole history into every turn. Start a fresh session when you
switch tasks rather than letting unrelated context accumulate.
